# Dataset Characteristics — Examples per Class & the Token-vs-Word Story

**Purpose.** This notebook extracts the dataset characteristics that defend the central thesis
claim — *token decoding is superior because it scales*. The narrative runs strongest-first:

| § | Section | The sentence it lets you say in the defense |
|---|---------|---------------------------------------------|
| 0 | Splits & determinism | Every split is fixed by seed 42 and reproducible; closed-vocab coverage is guaranteed. |
| 1 | Examples per class | The median word class has only a handful of examples; the median token class has thousands. |
| 2 | **Crossover figure** | As \|V\| grows, word examples/class collapse while token density is invariant (~39 classes). |
| 3 | OOV / coverage | A large fraction of test *words* are never seen in training; ~0% of test *tokens* are. |
| 4 | Distributional stability | The token label distribution is near-identical across splits; the word distribution is noisy. |
| 5 | CTC feasibility & supervision density | Both granularities satisfy the CTC length constraint; tokens give denser supervision. |
| 6 | Statistical power | The token test set has far more scoreable units, giving tighter error-rate CIs. |

**Scope (per the agreed plan).** DVSS, `dvss_parallel_only=True`. Because the parallel-matched
set has identical *text* for silent and voiced, every token-distribution metric (§1–§4, §6) is
**mode-independent**. Silent EMG is used only where duration matters (§5). The crossover figure
uses the existing `experiments/subsets/exp2_gradient` V1–V10 subsets as honest anchors.


## Setup


In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_repo_root(start=None):
    """Locate the repo root by walking up until data/extract.py and train.py are found.

    Makes the notebook robust to the working directory (Jupyter vs nbconvert)."""
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "data" / "extract.py").exists() and (cand / "train.py").exists():
            return cand
    raise RuntimeError(f"Could not locate repo root from {p}")


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "data"))

from extract import load_dataset           # noqa: E402
from transform import transform_dataset    # noqa: E402
from split import split_dataset, split_dvss_unseen_protocol  # noqa: E402

# Paths and constants tied to the real experiment configuration
SUBSET_DIR = REPO_ROOT / "experiments" / "subsets" / "exp2_gradient"
BANLIST_PATH = REPO_ROOT / "data" / "banlist.txt"
SAMPLE_RATE = 1000          # Hz  (features.sample_rate in base.yaml)
ENCODER_DOWNSAMPLE = 8     # downsampling factor of the calibrated neural acoustic encoder
SPLIT_SEED = 42             # the seed used throughout Prestudy 1/2 and Experiment 1

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
print("Repo root:", REPO_ROOT)
print("Subset dir exists:", SUBSET_DIR.exists(), "| Banlist exists:", BANLIST_PATH.exists())


### Load the parallel-matched DVSS metadata

All token-distribution metrics are computed on the **parallel-matched** utterance set
(`dvss_parallel_only=True`). Because silent and voiced share the same transcripts here, the
label statistics are identical for both modes — so §1–§4 and §6 are mode-independent. We sort by
`trial_id` before any split, exactly as `train.py` does, so the splits are deterministic.


In [ ]:
closed_meta = load_dataset("DVSS", mode="silent", vocabulary_type="closed_vocab", dvss_parallel_only=True)
open_meta = load_dataset("DVSS", mode="silent", vocabulary_type="open_vocab", dvss_parallel_only=True)

# Deterministic ordering before any split (matches train.py)
closed_meta = closed_meta.sort_values("trial_id").reset_index(drop=True)
open_meta = open_meta.sort_values("trial_id").reset_index(drop=True)

print(f"closed_vocab parallel silent : {len(closed_meta):>5} utterances")
print(f"open_vocab   parallel silent : {len(open_meta):>5} utterances")


### Helper metrics

`token_counts`, `token_entropy`, `token_balance_cv` are kept from the original notebook; the rest
extend them. A *class* is a token (39 intermediate token representation symbols) or a word type, depending on granularity.


In [ ]:
def token_counts(token_seqs):
    """Count occurrences of every token across a list of sequences."""
    counts = {}
    for seq in token_seqs:
        for token in seq:
            counts[token] = counts.get(token, 0) + 1
    return counts


def token_entropy(counts):
    """Shannon entropy (bits) over the token distribution."""
    if not counts:
        return 0.0
    total = sum(counts.values())
    p = np.array([c / total for c in counts.values()], dtype=float)
    return float(-(p * np.log2(p)).sum())


def effective_vocab(counts):
    """Perplexity 2**H: the number of equally-frequent classes that would match this entropy."""
    return float(2 ** token_entropy(counts))


def token_balance_cv(counts):
    """Coefficient of variation of class counts (0 = perfectly balanced)."""
    if not counts:
        return 0.0
    v = np.array(list(counts.values()), dtype=float)
    return float(v.std() / v.mean()) if v.mean() > 0 else 0.0


def gini(counts):
    """Gini coefficient of class counts (0 = balanced -> 1 = all mass in one class)."""
    if not counts:
        return 0.0
    x = np.sort(np.array(list(counts.values()), dtype=float))
    n = len(x)
    cum = np.cumsum(x)
    return float((n + 1 - 2 * (cum / cum[-1]).sum()) / n)


def frac_below(counts, k):
    """Fraction of classes that occur fewer than k times (k=2 -> singletons)."""
    if not counts:
        return 0.0
    v = np.array(list(counts.values()))
    return float((v < k).mean())


def vocab_of(df):
    """Set of token types present in a (split) dataframe's token_seq column."""
    return {t for seq in df["token_seq"] for t in seq}


def label_view(meta_df, token_based, load_emg=False):
    """transform_dataset wrapper. load_emg=False skips EMG I/O for fast label-only analysis."""
    mdf, vocab, _ = transform_dataset(meta_df, token_based=token_based, load_emg_data=load_emg)
    return mdf, vocab


# Four base label views, computed once and reused throughout the notebook.
closed_word_df, closed_word_vocab = label_view(closed_meta, token_based=False)
closed_tok_df, closed_tok_vocab = label_view(closed_meta, token_based=True)
open_word_df, open_word_vocab = label_view(open_meta, token_based=False)
open_tok_df, open_ph_vocab = label_view(open_meta, token_based=True)

print(f"closed : {len(closed_word_vocab):>4} word types | {len(closed_tok_vocab):>2} tokens")
print(f"open   : {len(open_word_vocab):>4} word types | {len(open_ph_vocab):>2} tokens")


## §0 — Splits & determinism

*Defends:* reproducibility, and that the closed-vocab split is not cherry-picked — every word
appears in both train and val by construction.

Three protocols are used across the thesis:
- **open 80/10/10** random split (`split_dataset`, seed 42) — Experiment 1 / RQ1.
- **closed 80/20** — the word baseline. Full coverage in *both* splits is structurally impossible
  (some words occur in a single trial and cannot land in both), so we guarantee full **train**
  coverage — every class learnable — and report the residual val gap, which equals the singleton count.
- **unseen-word (banlist)** — `split_dvss_unseen_protocol`, for compositional generalization.


In [ ]:
from collections import Counter


def split_coverage(model_df, all_words, seed):
    """(# words missing from train, # words missing from val) for an 80/20 split at this seed."""
    tr, va, _ = split_dataset(model_df, seed=seed, train_ratio=0.8, val_ratio=0.2, test_ratio=0.0)
    return len(all_words - vocab_of(tr)), len(all_words - vocab_of(va))


def train_coverage_seed(model_df, all_words, start=42, stop=200):
    """Smallest seed >= start with FULL train coverage (every class learnable)."""
    for s in range(start, stop):
        tr, _, _ = split_dataset(model_df, seed=s, train_ratio=0.8, val_ratio=0.2, test_ratio=0.0)
        if not (all_words - vocab_of(tr)):
            return s
    return None


# Words appearing in <2 trials cannot be in BOTH train and val -> structurally unsplittable.
trial_doc_count = Counter()
for seq in closed_word_df["token_seq"]:
    for w in set(seq):
        trial_doc_count[w] += 1
unsplittable = sorted(w for w, c in trial_doc_count.items() if c < 2)

closed_words = set(closed_word_vocab)
closed_seed = train_coverage_seed(closed_word_df, closed_words) or SPLIT_SEED
miss_tr, miss_va = split_coverage(closed_word_df, closed_words, closed_seed)
print(f"closed vocab: {len(closed_words)} word types; "
      f"{len(unsplittable)} occur in <2 trials (structurally unsplittable)")
print(f"train-coverage seed: {closed_seed}  ->  words missing from train: {miss_tr}, from val: {miss_va}")
print(f"(of the {miss_va} word(s) missing from val, {len(unsplittable)} are structurally unsplittable "
      f"(<2 trials) and the rest are seed-dependent; full both-split coverage is impossible)")

# Reproducibility: same seed -> identical trial-ID lists across independent calls.
a = split_dataset(open_word_df, seed=SPLIT_SEED)
b = split_dataset(open_word_df, seed=SPLIT_SEED)
for x, y, name in zip(a, b, ["train", "val", "test"]):
    assert list(x["trial_id"]) == list(y["trial_id"]), f"non-deterministic split: {name}"
print("Reproducibility: seed 42 -> identical trial-ID lists across re-runs OK")


In [ ]:
import contextlib
import io

# Summary table of every split protocol actually used in the experiments
o_tr, o_va, o_te = split_dataset(open_word_df, seed=SPLIT_SEED)
c_tr, c_va, _ = split_dataset(closed_word_df, seed=closed_seed, train_ratio=0.8, val_ratio=0.2, test_ratio=0.0)
with contextlib.redirect_stdout(io.StringIO()):  # silence the protocol's internal prints
    b_tr, b_va, b_te = split_dvss_unseen_protocol(open_word_df, banlist_path=str(BANLIST_PATH), seed=SPLIT_SEED)

splits_tbl = pd.DataFrame([
    dict(protocol="open 80/10/10 (random)", dataset="open_vocab", seed=SPLIT_SEED,
         train=len(o_tr), val=len(o_va), test=len(o_te), guarantee="-"),
    dict(protocol="closed 80/20 (train-coverage)", dataset="closed_vocab", seed=closed_seed,
         train=len(c_tr), val=len(c_va), test=0, guarantee="every word in train"),
    dict(protocol="unseen-word (banlist)", dataset="open_vocab", seed=SPLIT_SEED,
         train=len(b_tr), val=len(b_va), test=len(b_te), guarantee="test/val words unseen in train"),
])
splits_tbl


## §1 — Examples per class  *(the headline)*

*Defends:* word models have too many classes and too few examples each — token models never do.

Key numbers to quote: **median / min examples per class**, and the **fraction of word classes seen
fewer than 5 times** (a word the model essentially cannot learn). For tokens these pathologies
vanish. `effective_vocab` (= 2^entropy) and `gini` summarise how spread-out and learnable each
label space is.


In [ ]:
def class_metrics(mdf, granularity, name):
    counts = token_counts(mdf["token_seq"])
    v = np.array(list(counts.values()), dtype=float)
    total = float(v.sum())
    return dict(
        dataset=name,
        granularity=granularity,
        n_utterances=len(mdf),
        n_classes=len(counts),
        total_tokens=int(total),
        ex_per_class_median=float(np.median(v)),
        ex_per_class_min=float(v.min()),
        ex_per_class_mean=round(float(v.mean()), 1),
        singletons_pct=round(100 * frac_below(counts, 2), 1),
        below5_pct=round(100 * frac_below(counts, 5), 1),
        type_token_ratio=round(len(counts) / total, 4),
        entropy_bits=round(token_entropy(counts), 3),
        effective_vocab=round(effective_vocab(counts), 1),
        cv=round(token_balance_cv(counts), 3),
        gini=round(gini(counts), 3),
    )


ex_tbl = pd.DataFrame([
    class_metrics(closed_tok_df, "token", "closed"),
    class_metrics(closed_word_df, "word", "closed"),
    class_metrics(open_tok_df, "token", "open(full)"),
    class_metrics(open_word_df, "word", "open(full)"),
])
ex_tbl.set_index(["dataset", "granularity"]).T


In [ ]:
# Examples-per-class distribution: the heavy rare-class tail of words vs the dense token classes
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (mdf_tok, mdf_w, name) in zip(
        axes, [(closed_tok_df, closed_word_df, "closed"),
               (open_tok_df, open_word_df, "open(full)")]):
    for mdf, color, lab in [(mdf_w, "tab:red", "word"), (mdf_tok, "tab:blue", "token")]:
        v = np.array(list(token_counts(mdf["token_seq"]).values()), dtype=float)
        ax.hist(v, bins=np.logspace(0, np.log10(max(v.max(), 2)), 40), alpha=0.6, color=color, label=lab)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(name)
    ax.set_xlabel("examples per class")
    ax.legend()
axes[0].set_ylabel("# classes")
fig.suptitle("Examples-per-class distribution: words have a heavy rare-class tail; tokens do not")
plt.tight_layout()
plt.show()


## §2 — The crossover figure  *(centerpiece)*

*Defends:* the |V|^x research question, drawn entirely from your own data.

The `exp2_gradient` subsets hold utterance count and total duration roughly constant (~1 h, ~500–700
utterances) while the word vocabulary climbs. So they isolate the effect of vocabulary size: word
examples/class **must** fall as |V| rises, while the token inventory stays pinned at ~39 classes.
We anchor the curve with **closed(67)**, the **gradient V1–V10**, and **open(full)**. The 67 → ~1477
region has no data (different corpora), so we shade it honestly rather than interpolate.


In [ ]:
# Reuse the full-open token sequences for every subset (filter by trial_id) instead of re-transforming.
open_word_map = dict(zip(open_word_df["trial_id"], open_word_df["token_seq"]))
open_ph_map = dict(zip(open_tok_df["trial_id"], open_tok_df["token_seq"]))


def read_subset_ids(v):
    return [ln.strip() for ln in (SUBSET_DIR / f"V{v}_silent_trials.txt").read_text().splitlines() if ln.strip()]


def median_ex_per_class(seqs):
    c = token_counts(seqs)
    vals = list(c.values())
    return float(np.median(vals)), len(c), int(sum(vals))


cross_rows = []
# closed anchor
wm, wn, _ = median_ex_per_class(list(closed_word_df["token_seq"]))
pm, pn, _ = median_ex_per_class(list(closed_tok_df["token_seq"]))
cross_rows.append(dict(point=f"closed({wn})", V=wn, n_utt=len(closed_word_df),
                       word_med=wm, ph_med=pm, ph_classes=pn))
# gradient anchors
for v in range(1, 11):
    ids = read_subset_ids(v)
    wseqs = [open_word_map[i] for i in ids if i in open_word_map]
    pseqs = [open_ph_map[i] for i in ids if i in open_ph_map]
    wm, wn, _ = median_ex_per_class(wseqs)
    pm, pn, _ = median_ex_per_class(pseqs)
    cross_rows.append(dict(point=f"V{v}", V=wn, n_utt=len(ids),
                           word_med=wm, ph_med=pm, ph_classes=pn))
# open(full) anchor
wm, wn, _ = median_ex_per_class(list(open_word_df["token_seq"]))
pm, pn, _ = median_ex_per_class(list(open_tok_df["token_seq"]))
cross_rows.append(dict(point="open(full)", V=wn, n_utt=len(open_word_df),
                       word_med=wm, ph_med=pm, ph_classes=pn))

cross = pd.DataFrame(cross_rows).sort_values("V").reset_index(drop=True)
cross


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(cross["V"], cross["word_med"], "o-", color="tab:red", label="word: median examples / class")
ax.plot(cross["V"], cross["ph_med"], "s-", color="tab:blue", label="token: median examples / class")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("vocabulary size |V| (word types, log)")
ax.set_ylabel("median examples per class (log)")

# honest regime gap between closed(67) and the smallest gradient subset
grad_V = cross[cross["point"].str.match(r"V\d")]["V"]
closed_V = int(cross[cross["point"].str.startswith("closed")]["V"].iloc[0])
ax.axvspan(closed_V * 1.05, grad_V.min() * 0.95, color="grey", alpha=0.08)
ax.text(np.sqrt(closed_V * grad_V.min()), ax.get_ylim()[1] * 0.5, "regime gap\n(no data)",
        ha="center", va="top", fontsize=8, color="grey")

for _, r in cross.iterrows():
    if r["point"].startswith("closed") or r["point"] in ("open(full)", "V1", "V10"):
        ax.annotate(r["point"], (r["V"], r["word_med"]), textcoords="offset points",
                    xytext=(0, -14), fontsize=8, color="tab:red")
ax.set_title("Word examples/class collapse toward 1 as |V| grows;\n"
             "tokens keep 2-3 orders of magnitude more (only ~39 classes)")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Token-distribution stability across the gradient (D_KL vs full corpus) is already computed
# in the subset report -- overlay it to show the token label space barely moves as |V| scales.
report = pd.read_csv(SUBSET_DIR / "subset_report.csv")
report.columns = [c.strip() for c in report.columns]
vcol = "|V|" if "|V|" in report.columns else report.columns[1]
kcol = [c for c in report.columns if "D_KL" in c or c.strip() == "D_KL"][0]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(report[vcol], report[kcol], "d-", color="tab:green")
ax.set_xlabel("vocabulary size |V| (gradient subsets)")
ax.set_ylabel("$D_{KL}$(subset token dist || full corpus)")
ax.set_title("Token distribution is near-invariant across the vocabulary gradient")
plt.tight_layout()
plt.show()
report[[ "Subset", vcol, kcol]]


## §3 — OOV / coverage  *(the killer statistic)*

*Defends:* on open vocabulary a word model cannot even score a large share of the test set, while
a token model has full coverage.

We report the fraction of **test/val tokens** that were never seen in the training split. For words
this is large; for tokens it is ~0. We also show that even the 67-word closed vocabulary exercises
nearly the entire 39-token inventory, and that under the banlist protocol word OOV is ~100% by
construction while token coverage stays complete (compositional generalization is token-only).


In [ ]:
def oov_token_rate(train_seqs, eval_seqs):
    """Percent of tokens in eval whose type never appears in train."""
    train_vocab = set(token_counts(train_seqs).keys())
    tot = oov = 0
    for seq in eval_seqs:
        for t in seq:
            tot += 1
            oov += int(t not in train_vocab)
    return round(100 * oov / tot, 2) if tot else 0.0


oov_rows = []
for df, gran in [(open_word_df, "word"), (open_tok_df, "token")]:
    tr, va, te = split_dataset(df, seed=SPLIT_SEED)
    oov_rows.append(dict(granularity=gran,
                         val_oov_token_pct=oov_token_rate(tr["token_seq"], va["token_seq"]),
                         test_oov_token_pct=oov_token_rate(tr["token_seq"], te["token_seq"])))
oov_tbl = pd.DataFrame(oov_rows)
oov_tbl


In [ ]:
# Token-inventory coverage: even a tiny closed vocabulary trains nearly the full inventory.
closed_inv, open_inv = set(closed_tok_vocab), set(open_ph_vocab)
print(f"closed(67) exercises {len(closed_inv)} distinct tokens")
print(f"open(full) exercises {len(open_inv)} distinct tokens")
print("tokens in open but missing from closed:", sorted(open_inv - closed_inv) or "none")

# Banlist protocol: words in the unseen-word test set are OOV by construction; tokens are not.
print("\nUnseen-word (banlist) protocol -- test-set OOV token %:")
for df, gran in [(open_word_df, "word"), (open_tok_df, "token")]:
    with contextlib.redirect_stdout(io.StringIO()):  # silence the protocol's internal prints
        tr, va, te = split_dvss_unseen_protocol(df, banlist_path=str(BANLIST_PATH), seed=SPLIT_SEED)
    print(f"  {gran:<8}: {oov_token_rate(tr['token_seq'], te['token_seq'])}")


## §4 — Distributional stability across splits

*Defends:* the token label space is well-estimated and the train/val/test draws are representative;
the word label space is not.

Jensen–Shannon divergence (bits, symmetric, bounded) between the class distributions of the splits.
Small for tokens, much larger for words.


In [ ]:
def _dist(counts, support):
    total = sum(counts.values())
    return np.array([counts.get(t, 0) / total for t in support]) if total else np.zeros(len(support))


def js_divergence(c1, c2):
    """Jensen-Shannon divergence (bits) between two count dictionaries."""
    support = sorted(set(c1) | set(c2))
    p, q = _dist(c1, support), _dist(c2, support)
    m = 0.5 * (p + q)

    def kl(a, b):
        mask = a > 0
        return float(np.sum(a[mask] * np.log2(a[mask] / np.where(b[mask] > 0, b[mask], 1e-12))))

    return round(0.5 * kl(p, m) + 0.5 * kl(q, m), 4)


stab_rows = []
for df, gran in [(open_word_df, "word"), (open_tok_df, "token")]:
    tr, va, te = split_dataset(df, seed=SPLIT_SEED)
    ctr, cva, cte = (token_counts(tr["token_seq"]), token_counts(va["token_seq"]), token_counts(te["token_seq"]))
    stab_rows.append(dict(granularity=gran,
                          JS_train_val_bits=js_divergence(ctr, cva),
                          JS_train_test_bits=js_divergence(ctr, cte)))
pd.DataFrame(stab_rows)


## §5 — CTC feasibility & supervision density  *(silent EMG)*

*Defends:* the token-vs-word comparison is fair (both satisfy the CTC length constraint), and
tokens provide a denser training signal.

CTC requires the encoder output length `ceil(T / ENCODER_DOWNSAMPLE)` to be at least the target
length. We report the minimum margin and any violations per granularity, plus **labels per second**
(supervision density). EMG frame counts are read from the `.npy` headers via memory-mapping (cheap).


In [ ]:
def emg_num_frames(path):
    """Number of time frames T of an EMG file, without loading it fully (mmap header read)."""
    arr = np.load(path, mmap_mode="r")
    shp = arr.shape
    if len(shp) == 1:
        return int(shp[0])
    a, b = shp[0], shp[1]
    # ensure_time_major heuristic: (C, T) if first dim is the smaller and < 50
    return int(b) if (a < b and a < 50) else int(a)


frames = [emg_num_frames(p) for p in open_meta["emg_path"]]
dur = pd.DataFrame({"trial_id": open_meta["trial_id"].values, "T": frames})
dur["duration_s"] = dur["T"] / SAMPLE_RATE
dur["encoder_frames"] = np.ceil(dur["T"] / ENCODER_DOWNSAMPLE).astype(int)
print(f"EMG frames read for {len(dur)} silent utterances "
      f"(median {dur['duration_s'].median():.2f}s/utt)")


In [ ]:
ctc_rows = []
for df, gran in [(open_tok_df, "token"), (open_word_df, "word")]:
    m = df[["trial_id", "token_len"]].merge(dur, on="trial_id", how="inner")
    margin = m["encoder_frames"] - m["token_len"]
    ctc_rows.append(dict(
        granularity=gran,
        median_target_len=float(m["token_len"].median()),
        max_target_len=int(m["token_len"].max()),
        median_encoder_frames=float(m["encoder_frames"].median()),
        min_margin_frames=int(margin.min()),
        ctc_violations=int((margin < 0).sum()),
        labels_per_sec=round(float(m["token_len"].sum() / m["duration_s"].sum()), 2),
    ))
pd.DataFrame(ctc_rows)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for df, color, lab in [(open_word_df, "tab:red", "word"), (open_tok_df, "tab:blue", "token")]:
    ax.hist(df["token_len"], bins=40, alpha=0.6, color=color, label=lab)
ax.set_xlabel("target sequence length (tokens)")
ax.set_ylabel("# utterances")
ax.set_title("Token targets are longer -> denser supervision per utterance")
ax.legend()
plt.tight_layout()
plt.show()


## §6 — Statistical power of the test set

*Defends:* error-rate estimates on tokens are more precise simply because there are more scoreable
units, so the bootstrap CIs on TER are tighter than on WER.

For an illustrative error rate p = 0.30, the standard error of an utterance-pooled rate scales as
`sqrt(p(1-p)/N)` with N = number of scoreable tokens in the test split.


In [ ]:
p = 0.30  # illustrative error rate
pow_rows = []
for df, gran in [(open_tok_df, "token"), (open_word_df, "word")]:
    _, _, te = split_dataset(df, seed=SPLIT_SEED)
    n = int(sum(len(s) for s in te["token_seq"]))
    se = float(np.sqrt(p * (1 - p) / n))
    pow_rows.append(dict(granularity=gran, test_tokens=n,
                         se_at_p0_30=round(se, 5),
                         ci95_halfwidth_pp=round(100 * 1.96 * se, 3)))
pow_tbl = pd.DataFrame(pow_rows)
pow_tbl


## Appendix — Reference facts & original parallel-data note

The 39-token intermediate token representation inventory (stress stripped) is the entire token class space, regardless of
vocabulary. Below it is recomputed from the reference token set for reference, followed by the original note on why
`dvss_parallel_only` is required for a fair mode comparison.


In [ ]:
# [CENSORED / PROPRIETARY INTELLECTUAL PROPERTY: Intermediate discrete acoustic token inventory details withheld]
unique_tokens = [f'T{i+1:02d}' for i in range(39)]
print(f'Total unique intermediate tokens: {len(unique_tokens)}')
print(unique_tokens)


The voiced and silent recordings include parallel recordings captured in both speaking modes. These
amount to 1588 utterances recorded in both modes. Additionally there are non-parallel utterances
available only in the voiced mode, bringing the voiced total to 7065. For RQ1/RQ2 it would be unfair
to train the voiced model on more data overall, as that would blur the effect of speaking mode. Hence
`dvss_parallel_only=True` filters out the non-parallel utterances so the two modes share identical
transcripts and the mode effect is quantifiable.


In [ ]:
# Confirm parallel filtering equalises the two modes (and leaves silent unchanged).
for mode, parallel in [("silent", False), ("silent", True), ("voiced", False), ("voiced", True)]:
    n = len(load_dataset("DVSS", mode=mode, vocabulary_type="open_vocab", dvss_parallel_only=parallel))
    print(f"{mode:<7} parallel_only={str(parallel):<5} -> {n:>5} rows")
